# Adapter Analysis — Branch I (no S matrix)
Runs the full analysis pipeline from saved weights for the older model variant that uses a direct weight matrix `W` instead of `(g0, S)`.

**Inputs (all loaded from disk):**
- Weights: `W`, `om`
- Ratemaps: pre-computed `.npy` files (or recomputed from weights)
- Losses + lambdas (optional)
- Positions + representations for manifold (optional)

In [ ]:
import os, sys, pickle, json, logging, tempfile
import numpy as np
import umap
from sklearn.decomposition import PCA
from tqdm import tqdm
import mlflow
import plotly.graph_objects as go

# ── Add Pilot Decoder to path so analysis helpers are importable ───────────────
PILOT_DECODER = os.path.abspath('.')
if PILOT_DECODER not in sys.path:
    sys.path.insert(0, PILOT_DECODER)

from analysis import (
    frequency_plot,
    s_matrix_plot,
    loss_plots,
    quantitative_analysis,
    neuron_plotter_2d,
    module_ratemaps_plot,
    grid_type,
    manifold,
    manifold_cloud,
    manifold_slice,
    log_figure,
)
from scores import GridScorer

# ── Configuration ──────────────────────────────────────────────────────────────
FILEPATH = '/home/julian/Projects/MasterThesis/ICLR_Actionable_Reps/data/251106/170735'
COUNTER  = 0           # which saved checkpoint

GENERATE_MANIFOLD = True

sep = '_' if COUNTER != '' else ''
print(f'Run directory : {FILEPATH}')
print(f'Counter       : {COUNTER}')

## 1 · Load weights

In [ ]:
def _load_pkl(path):
    with open(path, 'rb') as f:
        obj = pickle.load(f)
    return np.array(obj)   # convert JAX arrays to numpy


def _pkl(name):
    return os.path.join(FILEPATH, f'{name}{sep}{COUNTER}.pkl')


# Prefer final weights if available
W  = _load_pkl(_pkl('W_final') if os.path.exists(_pkl('W_final')) else _pkl('W'))
om = _load_pkl(_pkl('om_final') if os.path.exists(_pkl('om_final')) else _pkl('om'))

D = W.shape[0]
M = om.shape[0]
print(f'D={D},  M={M}')
print(f'W : {W.shape},  om : {om.shape}')

## 2 · Load / compute ratemaps
Pre-saved `.npy` files are used when available; otherwise ratemaps are recomputed from `W` and `om` using a pure-NumPy implementation of `init_irreps_2D`.

In [ ]:
def init_irreps_2D_np(om: np.ndarray, phi: np.ndarray) -> np.ndarray:
    """NumPy port of ICLR `init_irreps_2D`.

    Args:
        om  : (M, 2) frequencies
        phi : (N, 2) positions

    Returns:
        I   : (2*M+1, N)  — [1, cos(k_1·phi), sin(k_1·phi), …, cos(k_M·phi), sin(k_M·phi)]
    """
    N = phi.shape[0]
    M_freq = om.shape[0]
    # k · phi  →  (M, N)
    k_dot_phi = (om @ phi.T)          # (M, N)
    I = np.ones((2 * M_freq + 1, N))
    I[1::2, :] = np.cos(k_dot_phi)   # rows 1, 3, 5, …
    I[2::2, :] = np.sin(k_dot_phi)   # rows 2, 4, 6, …
    return I


def get_ratemaps_np(W, om, res, widths):
    """Compute ratemaps, returning (D, res*res) arrays (flattened)."""
    maps = []
    for w in widths:
        xs  = np.linspace(-w / 2, w / 2, res)
        gx, gy = np.meshgrid(xs, xs)
        phi = np.stack([gx.ravel(), gy.ravel()], axis=1)   # (res^2, 2)
        I   = init_irreps_2D_np(om, phi)                   # (2*M+1, res^2)
        V   = W @ I                                         # (D, res^2)
        maps.append(V)
    return maps


res    = 70
widths = (1, 2, 4)
scale_tags = ('small', 'medium', 'large')

Vs = []
for tag, w in zip(scale_tags, widths):
    npy_path = os.path.join(FILEPATH, f'ratemaps_{tag}_{COUNTER}.npy')
    if os.path.exists(npy_path):
        V = np.load(npy_path)                    # (D, res, res)
        V = V.reshape(D, res * res)              # → (D, res*res)
        print(f'Loaded {tag} ratemaps from {npy_path}')
    else:
        print(f'Computing {tag} ratemaps (w={w})…')
        V = get_ratemaps_np(W, om, res, [w])[0]
    Vs.append(V)

V_small, V_medium, V_large = Vs
print(f'Ratemap shapes — small:{V_small.shape}  medium:{V_medium.shape}  large:{V_large.shape}')

## 3 · Load losses and lambdas (optional)

In [ ]:
# L has shape (4, T): rows are [total_loss, separation, positivity, norm]
L_path = _pkl('L')
L = _load_pkl(L_path) if os.path.exists(L_path) else None

lambda_pos_path  = _pkl('lambda_pos')
lambda_norm_path = _pkl('lambda_norm')
lambda_pos  = _load_pkl(lambda_pos_path)  if os.path.exists(lambda_pos_path)  else None
lambda_norm = _load_pkl(lambda_norm_path) if os.path.exists(lambda_norm_path) else None

if L is not None:
    print(f'L shape: {L.shape}  (rows: total, separation, positivity, norm)')
    train_losses = {
        'loss'       : L[0],
        'separation' : L[1],
        'positivity' : L[2],
        'norm'       : L[3],
    }
else:
    print('No loss file found — loss plots will be skipped.')
    train_losses = None

if lambda_pos is not None:
    print(f'lambda_pos shape: {lambda_pos.shape}')
if lambda_norm is not None:
    print(f'lambda_norm shape: {lambda_norm.shape}')

In [ ]:
# ── MLflow experiment + run ────────────────────────────────────────────────────
_sweep = os.path.basename(os.path.dirname(FILEPATH))   # e.g. 251106
_run   = os.path.basename(FILEPATH)                    # e.g. 170735
RUN_NAME = f'{_sweep}/{_run}/k{COUNTER}'

# Set to an existing run ID to resume logging, or None to create a new run
EXISTING_RUN_ID = "559475d91f57413492a74ac2a40b3986"

mlflow.set_tracking_uri(f"sqlite:///{PILOT_DECODER}/mlruns.db")
mlflow.set_experiment('adapter_analysis')

if EXISTING_RUN_ID is not None:
    active_run = mlflow.start_run(run_id=EXISTING_RUN_ID)
else:
    active_run = mlflow.start_run(run_name=RUN_NAME)
print(f'MLflow experiment : adapter_analysis')
print(f'Run name          : {RUN_NAME}')
print(f'Run ID            : {active_run.info.run_id}')

# Log provenance parameters
params_path = os.path.join(FILEPATH, 'parameters.json')
if os.path.exists(params_path):
    with open(params_path) as f:
        params = json.load(f)
    mlflow.log_params({k: v for k, v in params.items() if isinstance(v, (int, float, str, bool))})
mlflow.log_param('filepath', FILEPATH)
mlflow.log_param('counter', COUNTER)

## 4 · Load manifold data (optional)

In [ ]:
def _npy(name):
    return os.path.join(FILEPATH, f'{name}_{COUNTER}.npy')


positions_path       = _npy('positions_manifold')
representations_path = _npy('representations')

positions       = np.load(positions_path)       if os.path.exists(positions_path)       else None
representations = np.load(representations_path) if os.path.exists(representations_path) else None

if positions is not None and representations is not None:
    print(f'Positions       : {positions.shape}')        # (B, L, 2)
    print(f'Representations : {representations.shape}')  # (B, L, D)
else:
    print('Manifold data not found — manifold analysis will be skipped.')
    GENERATE_MANIFOLD = False

## 5 · Load UMAP embeddings from MLflow (optional)
Loads pre-computed UMAP embeddings saved by a prior manifold analysis run.
Skipped if `EXISTING_RUN_ID` is `None` or artifacts are absent.

In [ ]:
# ── Load UMAP embeddings from MLflow artifacts ─────────────────────────────────
mlflow_embeddings = {}  # module_idx -> {'embedding': ndarray, 'positions': ndarray}

if EXISTING_RUN_ID is not None:
    _client    = mlflow.MlflowClient()
    _artifacts = _client.list_artifacts(EXISTING_RUN_ID, path=f'manifold/k{COUNTER}')
    _names     = {a.path.split('/')[-1] for a in _artifacts}

    _module_idx = 0
    while True:
        emb_name = f'embedding_module{_module_idx}.npy'
        pos_name = f'positions_module{_module_idx}.npy'
        if emb_name not in _names:
            break
        emb_path = _client.download_artifacts(EXISTING_RUN_ID, f'manifold/k{COUNTER}/{emb_name}')
        pos_path = _client.download_artifacts(EXISTING_RUN_ID, f'manifold/k{COUNTER}/{pos_name}')
        mlflow_embeddings[_module_idx] = {
            'embedding':  np.load(emb_path),
            'positions':  np.load(pos_path),
        }
        _module_idx += 1

    if mlflow_embeddings:
        print(f'Loaded UMAP embeddings for {len(mlflow_embeddings)} module(s) from MLflow run {EXISTING_RUN_ID}')
        for idx, data in mlflow_embeddings.items():
            print(f'  module {idx}: embedding {data["embedding"].shape}, positions {data["positions"].shape}')
    else:
        print('No UMAP artifacts found for this run/counter.')
else:
    print('EXISTING_RUN_ID is None — skipping MLflow embedding load.')


---
## Analysis

### Frequency vectors

In [ ]:
fig_freq = frequency_plot(om)
log_figure(fig_freq, f'freq_plot_k{COUNTER}')
fig_freq.show()

### W matrix

In [ ]:
fig_W = go.Figure(go.Heatmap(
    z=W,
    colorscale='Viridis',
    colorbar=dict(tickfont=dict(size=20)),
))
fig_W.update_layout(
    title=dict(text='W matrix', font=dict(size=24)),
    xaxis=dict(constrain='domain', tickfont=dict(size=20), title='Irrep index', title_font=dict(size=20)),
    yaxis=dict(scaleanchor='x', scaleratio=1, constrain='domain', autorange='reversed',
               tickfont=dict(size=20), title='Neuron index', title_font=dict(size=20)),
    height=600, width=600,
)
log_figure(fig_W, f'W_matrix_k{COUNTER}')
fig_W.show()

### Loss curves

In [ ]:
if train_losses is not None:
    fig_loss = loss_plots(train_losses, lambda_pos=lambda_pos, lambda_norm=lambda_norm)
    log_figure(fig_loss, f'loss_curves_k{COUNTER}')
    fig_loss.show()
else:
    print('Skipped — no loss data.')

### Grid scores

In [ ]:
fig_scores_60, fig_scores_90, scores = quantitative_analysis(Vs, widths, res)
log_figure(fig_scores_60, f'grid_scores_60_k{COUNTER}')
log_figure(fig_scores_90, f'grid_scores_90_k{COUNTER}')
fig_scores_60.show()
fig_scores_90.show()

# Log scalar metrics
for key, value in scores.items():
    if isinstance(value, (int, float)):
        mlflow.log_metric(f'k{COUNTER}/{key}', value)

print(f"\nMax  grid scores — sm:{scores['sm_60_max']:.3f}  md:{scores['md_60_max']:.3f}  lg:{scores['lg_60_max']:.3f}")
print(f"Mean grid scores — sm:{scores['sm_60_mean']:.3f}  md:{scores['md_60_mean']:.3f}  lg:{scores['lg_60_mean']:.3f}")

### Neuron ratemaps

In [ ]:
for V, tag, score_key in [
    (V_small,  'small',  'sm_60'),
    (V_medium, 'medium', 'md_60'),
    (V_large,  'large',  'lg_60'),
]:
    fig = neuron_plotter_2d(V, res, scores[score_key])
    log_figure(fig, f'neurons_{tag}_k{COUNTER}_scores_60')
    fig.show()

for V, tag, score_key in [
    (V_small,  'small',  'sm_90'),
    (V_medium, 'medium', 'md_90'),
    (V_large,  'large',  'lg_90'),
]:
    fig = neuron_plotter_2d(V, res, scores[score_key])
    log_figure(fig, f'neurons_{tag}_k{COUNTER}_scores_90')
    fig.show()

### Clean ratemaps (no colorbar)

In [ ]:
for V, tag in [
    (V_small,  'small'),
    (V_medium, 'medium'),
    (V_large,  'large'),
]:
    fig = neuron_plotter_2d(V, res, show_colorbar=False)
    log_figure(fig, f'neurons_{tag}_k{COUNTER}_plain')
    fig.show()

### Module extraction and ratemaps

In [ ]:
large_width  = widths[2]
coord_range  = ((-large_width / 2, large_width / 2),) * 2
starts       = [0.2] * 10
ends         = np.linspace(0.4, 1.0, num=10)
scorer       = GridScorer(res, coord_range, zip(starts, ends.tolist()))

sacs   = [scorer.calculate_sac(V_large[i].reshape(res, res)) for i in range(D)]
labels, _ = scorer.get_modules(sacs, max_m=15)
labels = np.array(labels)
n_modules = int(labels.max()) + 1

print(f'{n_modules} modules found')
for m in range(n_modules):
    print(f'  Module {m}: {(labels == m).sum()} neurons')

In [ ]:
fig_mod = module_ratemaps_plot(V_large, res, scores['lg_60'], labels)
log_figure(fig_mod, f'module_ratemaps_k{COUNTER}')
fig_mod.show()

In [ ]:
fig_gt = grid_type(scores['lg_60'], scores['lg_90'], labels)
log_figure(fig_gt, f'grid_type_k{COUNTER}')
fig_gt.show()

### Manifold analysis

In [ ]:
if not GENERATE_MANIFOLD and not mlflow_embeddings:
    print('Manifold analysis skipped (data not available).')
elif representations is None and not mlflow_embeddings:
    print('No representations or cached embeddings — skipping.')
else:
    B, L_len, _ = representations.shape
    reps_flat   = representations.reshape(B * L_len, D)   # (B*L, D)
    flat_positions = positions.reshape(-1, 2) if positions is not None else None

    for module_idx in range(n_modules):
        module_mask = labels == module_idx
        module_reps = reps_flat[:, module_mask]            # (B*L, Dm)

        # ── Always compute PCA + scree ─────────────────────────────────────────
        pca  = PCA(n_components=0.95)
        pcs  = pca.fit_transform(module_reps)
        n_components = len(pca.explained_variance_ratio_)
        pc_numbers   = list(range(1, n_components + 1))
        import plotly.graph_objects as go
        scree_fig = go.Figure()
        scree_fig.add_trace(go.Bar(x=pc_numbers, y=pca.explained_variance_ratio_ * 100, name='Individual', opacity=0.7))
        scree_fig.add_trace(go.Scatter(x=pc_numbers, y=np.cumsum(pca.explained_variance_ratio_) * 100,
                                       mode='lines+markers', name='Cumulative',
                                       marker=dict(color='red'), line=dict(color='red')))
        scree_fig.update_layout(title=f'Scree Plot - Module {module_idx} ({n_components} components)',
                                xaxis_title='Principal Component', yaxis_title='Variance Explained (%)',
                                height=500, width=800)
        log_figure(scree_fig, f'scree_module{module_idx}_k{COUNTER}')

        # ── Use MLflow-cached embedding or run UMAP ────────────────────────────
        if module_idx in mlflow_embeddings:
            module_embedding = mlflow_embeddings[module_idx]['embedding']
            module_positions = mlflow_embeddings[module_idx]['positions']
            print(f'Module {module_idx}: loaded embedding {module_embedding.shape} from MLflow')
        elif GENERATE_MANIFOLD:
            reducer = umap.UMAP(n_components=3, metric='cosine', n_neighbors=500,
                                min_dist=0.8, init='spectral', n_jobs=24)
            module_embedding = reducer.fit_transform(pcs)
            module_positions = flat_positions
            with tempfile.TemporaryDirectory() as tmpdir:
                pos_path = f'{tmpdir}/positions_module{module_idx}.npy'
                emb_path = f'{tmpdir}/embedding_module{module_idx}.npy'
                np.save(pos_path, module_positions)
                np.save(emb_path, module_embedding)
                mlflow.log_artifact(pos_path, artifact_path=f'manifold/k{COUNTER}')
                mlflow.log_artifact(emb_path, artifact_path=f'manifold/k{COUNTER}')
        else:
            print(f'Module {module_idx}: no embedding available, skipping.')
            continue

        # ── Generate and log figures ───────────────────────────────────────────
        manifold_fig = manifold_cloud(module_embedding, module_positions, module_idx, n_components)
        slice_xy     = manifold_slice(module_embedding, (0, 1), module_idx, n_components)
        slice_xz     = manifold_slice(module_embedding, (0, 2), module_idx, n_components)
        slice_yz     = manifold_slice(module_embedding, (1, 2), module_idx, n_components)

        log_figure(scree_fig,    f'scree_module{module_idx}_k{COUNTER}')
        log_figure(manifold_fig, f'manifold_module{module_idx}_k{COUNTER}')
        log_figure(slice_xy,     f'manifold_slice_xy_module{module_idx}_k{COUNTER}')
        log_figure(slice_xz,     f'manifold_slice_xz_module{module_idx}_k{COUNTER}')
        log_figure(slice_yz,     f'manifold_slice_yz_module{module_idx}_k{COUNTER}')

        print(f'── Module {module_idx} ──────────────────────────────────')
        scree_fig.show()
        manifold_fig.show()
        slice_xy.show()
        slice_xz.show()
        slice_yz.show()


In [ ]:
mlflow.end_run()
print(f'Run {active_run.info.run_id} finished.')